In [53]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import pandas as pd
import pathlib
from sklearn.compose import ColumnTransformer

from clip_functions.data_clean import initialize_clip, data_clean, identify_default_images, assign_room_type, get_score, add_clip_columns, average_scoring


In [ ]:
data_path = pathlib.Path(".") / "raw_data"

# import images.csv
image_df = pd.read_csv(data_path / "images.csv")

# import listings.csv
listings_df = pd.read_csv(data_path / "listings.csv")

# TEMPORARY!!!
listings_df = listings_df[0:10]\
    .reset_index().drop(columns="index")
source_id = listings_df["source_id"]
image_df = image_df[image_df["source_id"].isin(source_id)]\
    .reset_index().drop(columns="index")

# run data cleaning function
#listings_df, image_df = data_clean(listings_df, image_df)

# define room list and attribute dict
RoomList = ["kitchen", "bathroom", "toilet", "living room", "bedroom", "walk-in closet", "closet", "entry inside", "exterior", "shop", "floor plan", "control panel", "entry outside", "blacony", "hallway"]
AttributeList = ["luxury", "brightness", "modernity"]

# initialize clip
clip = initialize_clip()

print("files imported, listings cleaned, clip initialized")

# add columns and remove unwanted images
image_df = add_clip_columns(df = image_df,
                            image_folder = data_path / "suumo_images",
                            room_list = RoomList,
                            attribute_list = AttributeList,
                            clip = clip)

# save csv
image_df.head()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


files imported, listings cleaned, clip initialized
start    source_id                                        listing_url  \
0   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
1   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
2   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
3   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
4   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   

                                           image_url               image_name  
0  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_c236d31dd0.jpg  
1  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_99ba43ab8d.jpg  
2  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_10972b9e12.jpg  
3  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_ec94786bdf.jpg  
4  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_67727e9e7f.jpg   161


/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


remove default    source_id                                        listing_url  \
0   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
1   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
2   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
3   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
4   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   

                                           image_url               image_name  \
0  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_c236d31dd0.jpg   
1  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_99ba43ab8d.jpg   
2  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_10972b9e12.jpg   
3  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_ec94786bdf.jpg   
4  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_67727e9e7f.jpg   

                                          image_path  default_image  
0  raw_data/suumo_images/

/home/bea/.pyenv/versions/clip-project/lib/python3.12/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


added room type    source_id                                        listing_url  \
0   20101205  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
1   20071087  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
2   20071087  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
3   20071087  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   
4   20071087  https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...   

                                           image_url               image_name  \
0  https://suumo.jp/front/gazo/bukken/030/N010000...  20101205_99ba43ab8d.jpg   
1  https://suumo.jp/front/gazo/bukken/030/N010000...  20071087_f70dacff50.jpg   
2  https://suumo.jp/front/gazo/bukken/030/N010000...  20071087_f2e5fe2057.jpg   
3  https://suumo.jp/front/gazo/bukken/030/N010000...  20071087_f91d0056e5.jpg   
4  https://suumo.jp/front/gazo/bukken/030/N010000...  20071087_12d994a609.jpg   

                                          image_path  default_image  \
0  raw_data/suumo_image

,source_id,listing_url,image_url,image_name,image_path,default_image,room_type,scoring_dict
0,20071087,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20071087_f70dacff50.jpg,raw_data/suumo_images/20071087/20071087_f70dac...,1,floor plan,"{'luxury': -1000, 'brightness': -1000, 'modern..."
1,20071087,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20071087_f2e5fe2057.jpg,raw_data/suumo_images/20071087/20071087_f2e5fe...,1,living room,"{'luxury': 0.34765625, 'brightness': 0.796875,..."
2,20071087,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20071087_f91d0056e5.jpg,raw_data/suumo_images/20071087/20071087_f91d00...,1,living room,"{'luxury': 0.4375, 'brightness': 0.62109375, '..."
3,20071087,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20071087_12d994a609.jpg,raw_data/suumo_images/20071087/20071087_12d994...,1,living room,"{'luxury': 0.34765625, 'brightness': 0.7539062..."
4,20071087,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,https://suumo.jp/front/gazo/bukken/030/N010000...,20071087_ae101558e7.jpg,raw_data/suumo_images/20071087/20071087_ae1015...,1,kitchen,"{'luxury': 0.3203125, 'brightness': 0.5625, 'm..."


In [ ]:
# scoring dict into column for each attribute
details_df = pd.json_normalize(image_df['scoring_dict'])
image_df = image_df.join(details_df)

# compute average score per room type and listing
average_scores = average_scoring(image_df, AttributeList)

# merge listings to include average scores per room type
data = listings_df.merge(average_scores, on = "source_id")
data.head()


average scores computed


In [49]:
get_score(str(pathlib.Path(".") / "raw_data/suumo_images/20013782/20013782_31d6d8fa9c.jpg"), "bathroom", AttributeList, clip)

{'luxury': 0.4375, 'brightness': 0.24609375, 'modernity': 0.5}

In [50]:
get_score(str(pathlib.Path(".") / "raw_data/suumo_images/20016425/20016425_b3d2d11f7d.jpg"), "bathroom", AttributeList, clip)

{'luxury': 0.10546875, 'brightness': 0.12109375, 'modernity': 0.40625}

In [51]:
get_score(str(pathlib.Path(".") / "raw_data/suumo_images/20007606/20007606_6bd0606793.jpg"), "bathroom", AttributeList, clip)

{'luxury': 0.26953125, 'brightness': 0.37890625, 'modernity': 0.81640625}

In [81]:
clip(image="https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_78385243", candidate_labels=RoomList)


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x7af72d253f60>